# Union `parking_v2_64m`

Vertically concatenate the four fiscal-year parking CSVs (2023–2026, ~17 GB total) into a single
table, keeping only the **38 attributes shared by all four** (listed in `intersected_attributes.txt`).
Outputs `merged.parquet` and `merged.csv`.

DuckDB does the heavy lifting: it streams the CSVs and spills to disk under a fixed memory budget, so
RAM stays bounded no matter how big the inputs are — nothing is ever fully materialised in memory.

Two header quirks are handled by matching columns on their **whitespace-stripped** name:
- 2026 has `"Days Parking In Effect    "` (trailing spaces) where the others have `"Days Parking In Effect"`.
- 2026 also carries an extra `Fiscal Year` column; it isn't in the intersected set, so it's dropped.

In [ ]:
from pathlib import Path

import duckdb

data_dir = Path("..") / "datasets" / "parking_v2_64m"
parquet_path = data_dir / "merged.parquet"
csv_path = data_dir / "merged.csv"

# scratch dir for DuckDB's on-disk spills (off the project tree)
temp_dir = Path(
    r"C:\Users\dvojk\AppData\Local\Temp\claude"
    r"\C--Users-dvojk-repositories-horizon-project\duckdb_spill"
)
temp_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")        # hard RAM ceiling; the rest spills to disk
con.execute("SET preserve_insertion_order = false")  # lets DuckDB stream the union cheaply
con.execute(f"SET temp_directory = '{temp_dir.as_posix()}'")
con.execute("SET max_temp_directory_size = '200GB'")

In [ ]:
# canonical 38 attributes: non-comment, non-blank lines of the intersected list
canon = [
    s
    for line in (data_dir / "intersected_attributes.txt").read_text(encoding="utf-8").splitlines()
    if (s := line.strip()) and not s.startswith("#")
]

files = sorted(data_dir.glob("Parking_Violations_*.csv"))
print(f"{len(canon)} intersected attributes, {len(files)} source files")
for f in files:
    print(" ", f.name)

In [ ]:
# Build a per-file SELECT that pulls the 38 canonical columns, aliasing each file's
# actual header back to the canonical name. Matching on the stripped name lets the alias
# line columns up by name across all four files even where headers differ cosmetically.


def header_of(path: Path) -> list[str]:
    # Ask DuckDB for the header so names match exactly what the COPY query binds against —
    # DuckDB trims trailing whitespace from headers, so "Days Parking In Effect    " arrives
    # here already as "Days Parking In Effect".
    rows = con.execute(
        f"DESCRIBE SELECT * FROM read_csv('{path.as_posix()}', all_varchar = true, header = true)"
    ).fetchall()
    return [r[0] for r in rows]


def select_for(path: Path) -> str:
    by_stripped = {h.strip(): h for h in header_of(path)}
    missing = [c for c in canon if c not in by_stripped]
    if missing:
        raise ValueError(f"{path.name} is missing canonical columns: {missing}")
    cols = ",\n        ".join(f'"{by_stripped[c]}" AS "{c}"' for c in canon)
    return (
        f"SELECT\n        {cols}\n"
        f"    FROM read_csv('{path.as_posix()}', all_varchar = true, header = true)"
    )


# all_varchar: read every column as text. The inputs are dirty (FD violations) and FD repair
# treats values categorically, so type inference would only get in the way — same choice as
# csv_to_parquet.ipynb.
union_sql = "\nUNION ALL BY NAME\n".join(select_for(p) for p in files)
print(union_sql[:1200], "...")

In [ ]:
# Pass 1: stream the four CSVs through the union straight into one Parquet file.
# Reads ~17 GB of CSV exactly once. zstd + 1M-row groups mirror csv_to_parquet.ipynb.
con.execute(
    f"""
    COPY ({union_sql})
    TO '{parquet_path.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)
    """
)
pq_mb = parquet_path.stat().st_size / 1024 / 1024
print(f"{parquet_path.name}: {pq_mb:,.1f} MB")

In [ ]:
# Pass 2: emit the CSV from the Parquet (cheap re-read) rather than re-scanning the 17 GB
# of source CSVs. The output CSV is large — it's the full union — so expect tens of GB.
con.execute(
    f"""
    COPY (SELECT * FROM read_parquet('{parquet_path.as_posix()}'))
    TO '{csv_path.as_posix()}'
    (FORMAT CSV, HEADER)
    """
)
csv_gb = csv_path.stat().st_size / 1024 / 1024 / 1024
print(f"{csv_path.name}: {csv_gb:,.2f} GB")

In [ ]:
# sanity check: row/col counts and a peek at the merged Parquet
n_rows = con.execute(
    f"SELECT count(*) FROM read_parquet('{parquet_path.as_posix()}')"
).fetchone()[0]
n_cols = len(con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{parquet_path.as_posix()}')"
).fetchall())
print(f"{n_rows:,} rows, {n_cols} cols (expected {len(canon)})")
con.execute(f"SELECT * FROM read_parquet('{parquet_path.as_posix()}') LIMIT 5").df()